# Setup

In [ ]:
from truveta.study import Client, OutputMode, display_df
import pyspark.pandas as ps
import pandas as pd
import numpy as np

In [ ]:
client = Client(output_mode = OutputMode.PandasOnSpark)
study = client.get_study()

In [ ]:
population = study.get_population(title = "Aim1b_Female_Update")

In [ ]:
snapshot = population.get_latest_snapshot()

In [ ]:
snapshot.get_tables()

# First Vascular Event

In [ ]:
query = """
SELECT PersonId, IndexDateTime, firstVascularEventTime
FROM FeatureTable_FirstVascularEventTable
"""

FirstVascularEvent = spark.sql(query)
save_path = "/output_data/Male_FirstVascularEvent"
study.save_artifacts_data(df=FirstVascularEvent, path=save_path, data_type="parquet", mode="overwrite")

In [ ]:
path_df1 = "/output_data/Female_FirstVascularEvent"
Female_FirstVascular_Event = study.load_artifacts_data(path_df1)

path_df2 = "/output_data/Male_FirstVascularEvent"
Male_FirstVascularEvent = study.load_artifacts_data(path_df2)

# concat
combined_df = ps.concat([Female_FirstVascular_Event, Male_FirstVascularEvent])

# distinct
combined_df = combined_df.drop_duplicates()

study.save_artifacts_data(df=combined_df, path="/output_data/All_sex_FirstVascularEvent", data_type="parquet", mode="overwrite")

# Data Cleaning

## Create base tables

In [ ]:
query = """
WITH outcome_bmi AS (
    SELECT PersonId, IndexDateTime, MeasureDateTime, BmiValue
    FROM (
        SELECT
            PersonId,
            IndexDateTime,
            BmiTime AS MeasureDateTime,
            BmiValue,
            ROW_NUMBER() OVER (
                PARTITION BY PersonId, IndexDateTime
                ORDER BY ABS(DATEDIFF(BmiTime, DATE_ADD(IndexDateTime, 180))) ASC
            ) AS rn
        FROM FeatureTable_OutcomeBmiTable
    ) t
    WHERE rn = 1
),
outcome_weight AS (
    SELECT PersonId, IndexDateTime, MeasureDateTime, WeightValue
    FROM (
        SELECT
            PersonId,
            IndexDateTime,
            WeightTime AS MeasureDateTime,
            WeightValue,
            ROW_NUMBER() OVER (
                PARTITION BY PersonId, IndexDateTime
                ORDER BY ABS(DATEDIFF(WeightTime, DATE_ADD(IndexDateTime, 180))) ASC
            ) AS rn
        FROM FeatureTable_OutcomeWeightTable
    ) t
    WHERE rn = 1
),
outcome_hba1c AS (
    SELECT PersonId, IndexDateTime, MeasureDateTime, HbA1cValue
    FROM (
        SELECT
            PersonId,
            IndexDateTime,
            HbA1cTime AS MeasureDateTime,
            HbA1cValue,
            ROW_NUMBER() OVER (
                PARTITION BY PersonId, IndexDateTime
                ORDER BY ABS(DATEDIFF(HbA1cTime, DATE_ADD(IndexDateTime, 180))) ASC
            ) AS rn
        FROM FeatureTable_OutcomeHbA1cTable
    ) t
    WHERE rn = 1
),
outcome_hdl AS (
    SELECT PersonId, IndexDateTime, MeasureDateTime, HdlCholesterolValue
    FROM (
        SELECT
            PersonId,
            IndexDateTime,
            HdlCholesterolTime AS MeasureDateTime,
            HdlCholesterolValue,
            ROW_NUMBER() OVER (
                PARTITION BY PersonId, IndexDateTime
                ORDER BY ABS(DATEDIFF(HdlCholesterolTime, DATE_ADD(IndexDateTime, 180))) ASC
            ) AS rn
        FROM FeatureTable_OutcomeHdlCholesterolTable
    ) t
    WHERE rn = 1
),
outcome_ldl AS (
    SELECT PersonId, IndexDateTime, MeasureDateTime, LdlCholesterolValue
    FROM (
        SELECT
            PersonId,
            IndexDateTime,
            LdlCholesterolTime AS MeasureDateTime,
            LdlCholesterolValue,
            ROW_NUMBER() OVER (
                PARTITION BY PersonId, IndexDateTime
                ORDER BY ABS(DATEDIFF(LdlCholesterolTime, DATE_ADD(IndexDateTime, 180))) ASC
            ) AS rn
        FROM FeatureTable_OutcomeLdlCholesterolTable
    ) t
    WHERE rn = 1
),
outcome_tg AS (
    SELECT PersonId, IndexDateTime, MeasureDateTime, SerumTriglyceridesValue
    FROM (
        SELECT
            PersonId,
            IndexDateTime,
            SerumTriglyceridesTime AS MeasureDateTime,
            SerumTriglyceridesValue,
            ROW_NUMBER() OVER (
                PARTITION BY PersonId, IndexDateTime
                ORDER BY ABS(DATEDIFF(SerumTriglyceridesTime, DATE_ADD(IndexDateTime, 180))) ASC
            ) AS rn
        FROM FeatureTable_OutcomeSerumTriglyceridesTable
    ) t
    WHERE rn = 1
),
outcome_sbp AS (
    SELECT PersonId, IndexDateTime, MeasureDateTime, SystolicBloodPressureValue
    FROM (
        SELECT
            PersonId,
            IndexDateTime,
            SystolicBloodPressureTime AS MeasureDateTime,
            SystolicBloodPressureValue,
            ROW_NUMBER() OVER (
                PARTITION BY PersonId, IndexDateTime
                ORDER BY ABS(DATEDIFF(SystolicBloodPressureTime, DATE_ADD(IndexDateTime, 180))) ASC
            ) AS rn
        FROM FeatureTable_OutcomeSystolicBloodPressureTable
    ) t
    WHERE rn = 1
)

SELECT
    o.*,

    -- baseline
    alt.MeasureDateTime AS Alt_MeasureDateTime, alt.AltValue, alt.Unit AS Alt_Unit,
    ast.MeasureDateTime AS Ast_MeasureDateTime, ast.AstValue, ast.Unit AS Ast_Unit,
    bhcg.MeasureDateTime AS BetaHcg_MeasureDateTime, bhcg.BetaHcgValue, bhcg.Unit AS BetaHcg_Unit,
    bmi.MeasureDateTime AS Bmi_MeasureDateTime, bmi.BmiValue,
    crp.MeasureDateTime AS CRP_MeasureDateTime, crp.CReactiveProteinValue, crp.Unit AS CRP_Unit,
    dbp.MeasureDateTime AS DBP_MeasureDateTime, dbp.DiastolicBloodPressureValue, dbp.Unit AS DBP_Unit,
    egfr.MeasureDateTime AS EGFR_MeasureDateTime, egfr.EGfrValue, egfr.Unit AS EGFR_Unit,
    hba1c.MeasureDateTime AS HbA1c_MeasureDateTime, hba1c.HbA1cValue, hba1c.Unit AS HbA1c_Unit,
    hscrp.MeasureDateTime AS HsCRP_MeasureDateTime, hscrp.HsCRPValue, hscrp.Unit AS HsCRP_Unit,
    hdl.MeasureDateTime AS HDL_MeasureDateTime, hdl.HdlCholesterolValue, hdl.Unit AS HDL_Unit,
    ht.MeasureDateTime AS Height_MeasureDateTime, ht.HeightValue, ht.Unit AS Height_Unit,
    lpa.MeasureDateTime AS LpA_MeasureDateTime, lpa.LipoproteinAValue, lpa.Unit AS LpA_Unit,
    scr.MeasureDateTime AS Creatinine_MeasureDateTime, scr.SerumCreatinineValue, scr.Unit AS Creatinine_Unit,
    tg.MeasureDateTime AS TG_MeasureDateTime, tg.SerumTriglyceridesValue, tg.Unit AS TG_Unit,
    sbp.MeasureDateTime AS SBP_MeasureDateTime, sbp.SystolicBloodPressureValue, sbp.Unit AS SBP_Unit,
    chol.MeasureDateTime AS Chol_MeasureDateTime, chol.TotalCholesterolValue, chol.Unit AS Chol_Unit,
    wbc.MeasureDateTime AS WBC_MeasureDateTime, wbc.WhiteBloodCellCountValue, wbc.Unit AS WBC_Unit,
    wt.MeasureDateTime AS Weight_MeasureDateTime, wt.WeightValue, wt.Unit AS Weight_Unit,
    p.MeasureDateTime AS PlateletCount_MeasureDateTime, p.PlateletCountValue, p.Unit AS PlateletCount_Unit,
    hemo.MeasureDateTime AS Hemoglobin_MeasureDateTime, hemo.HemoglobinValue, hemo.Unit AS Hemoglobin_Unit,
    ldl.MeasureDateTime AS LDL_MeasureDateTime, ldl.LdlCholesterolValue, ldl.Unit AS LDL_Unit,

    -- outcome
    obmi.MeasureDateTime AS Outcome_Bmi_MeasureDateTime, obmi.BmiValue AS Outcome_BmiValue,
    owt.MeasureDateTime  AS Outcome_Weight_MeasureDateTime, owt.WeightValue AS Outcome_WeightValue,
    ohba1c.MeasureDateTime AS Outcome_HbA1c_MeasureDateTime, ohba1c.HbA1cValue AS Outcome_HbA1cValue,
    ohdl.MeasureDateTime AS Outcome_HDL_MeasureDateTime, ohdl.HdlCholesterolValue AS Outcome_HDLValue,
    oldl.MeasureDateTime AS Outcome_LDL_MeasureDateTime, oldl.LdlCholesterolValue AS Outcome_LDLValue,
    otg.MeasureDateTime  AS Outcome_TG_MeasureDateTime, otg.SerumTriglyceridesValue AS Outcome_TGValue,
    osbp.MeasureDateTime AS Outcome_SBP_MeasureDateTime, osbp.SystolicBloodPressureValue AS Outcome_SBPValue

FROM FeatureTable_Aim1bEntirePopulation o
-- baseline
LEFT JOIN FeatureTable_BaselineAltTable alt  ON o.PersonId = alt.PersonId AND o.FirstMedTime = alt.IndexDateTime
LEFT JOIN FeatureTable_BaselineAstTable ast  ON o.PersonId = ast.PersonId AND o.FirstMedTime = ast.IndexDateTime
LEFT JOIN FeatureTable_BaselineBetaHcgTable bhcg ON o.PersonId = bhcg.PersonId AND o.FirstMedTime = bhcg.IndexDateTime
LEFT JOIN FeatureTable_BaselineBmiTable bmi  ON o.PersonId = bmi.PersonId AND o.FirstMedTime = bmi.IndexDateTime
LEFT JOIN FeatureTable_BaselineCReactiveProteinTable crp ON o.PersonId = crp.PersonId AND o.FirstMedTime = crp.IndexDateTime
LEFT JOIN FeatureTable_BaselineDiastolicBloodPressureTable dbp ON o.PersonId = dbp.PersonId AND o.FirstMedTime = dbp.IndexDateTime
LEFT JOIN FeatureTable_BaselineEGfrTable egfr ON o.PersonId = egfr.PersonId AND o.FirstMedTime = egfr.IndexDateTime
LEFT JOIN FeatureTable_BaselineHbA1cTable hba1c ON o.PersonId = hba1c.PersonId AND o.FirstMedTime = hba1c.IndexDateTime
LEFT JOIN FeatureTable_BaselineHsCRPTable hscrp ON o.PersonId = hscrp.PersonId AND o.FirstMedTime = hscrp.IndexDateTime
LEFT JOIN FeatureTable_BaselineHdlCholesterolTable hdl ON o.PersonId = hdl.PersonId AND o.FirstMedTime = hdl.IndexDateTime
LEFT JOIN FeatureTable_BaselineHeightTable ht ON o.PersonId = ht.PersonId AND o.FirstMedTime = ht.IndexDateTime
LEFT JOIN FeatureTable_BaselineLipoproteinATable lpa ON o.PersonId = lpa.PersonId AND o.FirstMedTime = lpa.IndexDateTime
LEFT JOIN FeatureTable_BaselineSerumCreatinineTable scr ON o.PersonId = scr.PersonId AND o.FirstMedTime = scr.IndexDateTime
LEFT JOIN FeatureTable_BaselineSerumTriglyceridesTable tg ON o.PersonId = tg.PersonId AND o.FirstMedTime = tg.IndexDateTime
LEFT JOIN FeatureTable_BaselineSystolicBloodPressureTable sbp ON o.PersonId = sbp.PersonId AND o.FirstMedTime = sbp.IndexDateTime
LEFT JOIN FeatureTable_BaselineTotalCholesterolTable chol ON o.PersonId = chol.PersonId AND o.FirstMedTime = chol.IndexDateTime
LEFT JOIN FeatureTable_BaselineWbcTable wbc ON o.PersonId = wbc.PersonId AND o.FirstMedTime = wbc.IndexDateTime
LEFT JOIN FeatureTable_BaselineWeightTable wt ON o.PersonId = wt.PersonId AND o.FirstMedTime = wt.IndexDateTime
LEFT JOIN FeatureTable_BaselinePlateletCountTable p ON o.PersonId = p.PersonId AND o.FirstMedTime = p.IndexDateTime
LEFT JOIN FeatureTable_BaselineHemoglobinTable hemo on o.PersonId = hemo.PersonId AND o.FirstMedTime = hemo.IndexDateTime
LEFT JOIN FeatureTable_BaselineLdlCholesterolTable ldl on o.PersonId = ldl.PersonId AND o.FirstMedTime = ldl.IndexDateTime

-- outcome
LEFT JOIN outcome_bmi obmi   ON o.PersonId = obmi.PersonId   AND o.FirstMedTime = obmi.IndexDateTime
LEFT JOIN outcome_weight owt ON o.PersonId = owt.PersonId   AND o.FirstMedTime = owt.IndexDateTime
LEFT JOIN outcome_hba1c ohba1c ON o.PersonId = ohba1c.PersonId AND o.FirstMedTime = ohba1c.IndexDateTime
LEFT JOIN outcome_hdl ohdl   ON o.PersonId = ohdl.PersonId  AND o.FirstMedTime = ohdl.IndexDateTime
LEFT JOIN outcome_ldl oldl   ON o.PersonId = oldl.PersonId  AND o.FirstMedTime = oldl.IndexDateTime
LEFT JOIN outcome_tg otg     ON o.PersonId = otg.PersonId   AND o.FirstMedTime = otg.IndexDateTime
LEFT JOIN outcome_sbp osbp   ON o.PersonId = osbp.PersonId  AND o.FirstMedTime = osbp.IndexDateTime
"""

Aim1b_male_basedata = spark.sql(query)

# save_path = "/output_data/T2DM_Female_basedata_Mar08"
# study.save_artifacts_data(df=Aim1b_Female_basedata, path=save_path, data_type="parquet", mode="overwrite")

save_path = "/output_data/T2DM_male_basedata_Mar09"
study.save_artifacts_data(df=Aim1b_male_basedata, path=save_path, data_type="parquet", mode="overwrite")

In [ ]:
from pyspark.sql.functions import col, months_between

# T2D Cohort generation
basedata = study.load_artifacts_data("/output_data/T2DM_male_basedata_Mar09")
basedata_spark = basedata.to_spark()

base_filtered_df = basedata_spark.filter(
    (col("ExclT2d") == 'true') &
    ((months_between(col("FirstMedTime"), col("BirthDateTime")) / 12) >= 18) &
    (col("FirstMedGroup").isin('glp', 'SGLT', 'DPP4')) &
    (col("FirstMedTime").isNotNull()) &
    (col("HasEncounter") == 'true') &
    # exclusion criteria
    (col("ExclT1d") == 'false') &
    (col("ExclGestDiab") == 'false') &
    (col("ExclMody") == 'false') &
    (col("ExclMedullaryThyroidCarcinoma") == 'false') &
    (col("ExclPancreaticCancer") == 'false') &
    (col("ExclChronicPancreatitis") == 'false') &
    (col("ExclEndStageRenalDisease") == 'false')
)

# HasCvd == true
cvd_df = base_filtered_df.filter(col("HasCvd") == 'true').drop_duplicates()

# HasCvd == false
# no_cvd_df = base_filtered_df.filter(col("HasCvd") == 'false').drop_duplicates()

# Save data
save_path = "/output_data/T2DM_male_secondary_basedata_Mar09"
study.save_artifacts_data(df=cvd_df, path=save_path, data_type="parquet", mode="overwrite")

# Save data
# save_path = "/output_data/T2DM_male_primary_basedata_Mar09"
# study.save_artifacts_data(df=no_cvd_df, path=save_path, data_type="parquet", mode="overwrite")

In [ ]:
from pyspark.sql.functions import col, months_between

# Obese Cohort generation
basedata = study.load_artifacts_data("/output_data/T2DM_Female_basedata_Mar08")
basedata_spark = basedata.to_spark()

base_filtered_df = basedata_spark.filter(
    (col("BmiValue") >= 27) &
    (col("ExclT2d") == 'false') &
    (col("ExclPriorDiabeticDrugs") == 'false') &
    ((months_between(col("FirstMedTime"), col("BirthDateTime")) / 12) >= 18) &
    (col("FirstMedGroup").isin('glp', 'NB', 'PT')) &
    (col("FirstMedTime").isNotNull()) &
    (col("HasEncounter") == 'true') &
    # exclusion criteria
    (col("ExclT1d") == 'false') &
    (col("ExclGestDiab") == 'false') &
    (col("ExclMody") == 'false') &
    (col("ExclMedullaryThyroidCarcinoma") == 'false') &
    (col("ExclPancreaticCancer") == 'false') &
    (col("ExclChronicPancreatitis") == 'false') &
    (col("ExclEndStageRenalDisease") == 'false')
)

# # HasCvd == true
cvd_df = base_filtered_df.filter(col("HasCvd") == 'true').drop_duplicates()

# HasCvd == false
# no_cvd_df = base_filtered_df.filter(col("HasCvd") == 'false').drop_duplicates()

# Save data
save_path = "/output_data/Obese_female_secondary_basedata_Mar09"
study.save_artifacts_data(df=cvd_df, path=save_path, data_type="parquet", mode="overwrite")

# Save data
# save_path = "/output_data/Obese_female_primary_basedata_Mar09"
# study.save_artifacts_data(df=no_cvd_df, path=save_path, data_type="parquet", mode="overwrite")

In [ ]:
path_df1 = "/output_data/Obese_female_secondary_basedata_Apr17"
Female_basedata = study.load_artifacts_data(path_df1)

path_df2 = "/output_data/Obese_male_secondary_basedata_Apr17"
Male_basedata = study.load_artifacts_data(path_df2)

# concat
combined_df = ps.concat([Female_basedata, Male_basedata])

# distinct
combined_df = combined_df.drop_duplicates()

study.save_artifacts_data(df=combined_df, path="/output_data/All_sex_Obese_secondary_basedata_Apr17", data_type="parquet", mode="overwrite")

## Missing rate check

In [ ]:
# final secondary merged data
secondary_basedata = study.load_artifacts_data("/output_data/All_sex_T2DM_secondary_basedata_Mar09")

total_count = len(secondary_basedata)

missing_result = (
    secondary_basedata.isnull().sum() / total_count * 100
).round(2).to_frame(name="Missing_Rate(%)")

missing_result.index.name = "Column"

print(f"Total number of patients: {total_count:,}")
print(missing_result.sort_values("Missing_Rate(%)", ascending=False).to_string())

In [ ]:
# final primary merged data
primary_basedata = study.load_artifacts_data("/output_data/All_sex_T2DM_primary_basedata_Mar09")

total_count = len(primary_basedata)

missing_result = (
    primary_basedata.isnull().sum() / total_count * 100
).round(2).to_frame(name="Missing_Rate(%)")

missing_result.index.name = "Column"

print(f"Total number of patients: {total_count:,}")
print(missing_result.sort_values("Missing_Rate(%)", ascending=False).to_string())

In [ ]:
# final secondary merged data
secondary_basedata = study.load_artifacts_data("/output_data/All_sex_Obese_secondary_basedata_Apr17")

total_count = len(secondary_basedata)

missing_result = (
    secondary_basedata.isnull().sum() / total_count * 100
).round(2).to_frame(name="Missing_Rate(%)")

missing_result.index.name = "Column"

print(f"Total number of patients: {total_count:,}")
print(missing_result.sort_values("Missing_Rate(%)", ascending=False).to_string())

In [ ]:
# final secondary merged data
primary_basedata = study.load_artifacts_data("/output_data/All_sex_Obese_primary_basedata_Apr17")

total_count = len(secondary_basedata)

missing_result = (
    primary_basedata.isnull().sum() / total_count * 100
).round(2).to_frame(name="Missing_Rate(%)")

missing_result.index.name = "Column"

print(f"Total number of patients: {total_count:,}")
print(missing_result.sort_values("Missing_Rate(%)", ascending=False).to_string())

In [ ]:
t2dm_female_basedata = study.load_artifacts_data("/output_data/T2DM_female_secondary_basedata_Mar08")

total_count = len(t2dm_female_basedata)

missing_result = (
    t2dm_female_basedata.isnull().sum() / total_count * 100
).round(2).to_frame(name="Missing_Rate(%)")

missing_result.index.name = "Column"

print(f"Total number of patients: {total_count:,}")
print(missing_result.sort_values("Missing_Rate(%)", ascending=False).to_string())

In [ ]:
t2dm_female_basedata = study.load_artifacts_data("/output_data/T2DM_female_primary_basedata_Mar08")

total_count = len(t2dm_female_basedata)

missing_result = (
    t2dm_female_basedata.isnull().sum() / total_count * 100
).round(2).to_frame(name="Missing_Rate(%)")

missing_result.index.name = "Column"

print(f"Total number of patients: {total_count:,}")
print(missing_result.sort_values("Missing_Rate(%)", ascending=False).to_string())

In [ ]:
t2dm_male_basedata = study.load_artifacts_data("/output_data/T2DM_male_secondary_basedata_Mar09")

total_count = len(t2dm_male_basedata)

missing_result = (
    t2dm_male_basedata.isnull().sum() / total_count * 100
).round(2).to_frame(name="Missing_Rate(%)")

missing_result.index.name = "Column"

print(f"Total number of patients: {total_count:,}")
print(missing_result.sort_values("Missing_Rate(%)", ascending=False).to_string())

In [ ]:
t2dm_male_basedata = study.load_artifacts_data("/output_data/T2DM_male_primary_basedata_Mar09")

total_count = len(t2dm_male_basedata)

missing_result = (
    t2dm_male_basedata.isnull().sum() / total_count * 100
).round(2).to_frame(name="Missing_Rate(%)")

missing_result.index.name = "Column"

print(f"Total number of patients: {total_count:,}")
print(missing_result.sort_values("Missing_Rate(%)", ascending=False).to_string())

## Lab measurements value check

In [ ]:
# Final merged secondary data
t2dm_secondary_basedata = study.load_artifacts_data("/output_data/All_sex_T2DM_secondary_basedata_Mar09")
value_cols = [c for c in t2dm_secondary_basedata.columns if 'Value' in c]

# Q1, Median, Q3, Min, Max
desc = t2dm_secondary_basedata[value_cols].describe(percentiles=[0.25, 0.5, 0.75])

result = desc.loc[['min', '25%', '50%', '75%', 'max']]
result.index = ['Min', 'Q1', 'Median', 'Q3', 'Max']

print(result.T.to_string())

In [ ]:
# Final merged primary data
t2dm_primary_basedata = study.load_artifacts_data("/output_data/All_sex_T2DM_primary_basedata_Mar09")
value_cols = [c for c in t2dm_primary_basedata.columns if 'Value' in c]

# Q1, Median, Q3, Min, Max
desc = t2dm_primary_basedata[value_cols].describe(percentiles=[0.25, 0.5, 0.75])

result = desc.loc[['min', '25%', '50%', '75%', 'max']]
result.index = ['Min', 'Q1', 'Median', 'Q3', 'Max']

print(result.T.to_string())

In [ ]:
# Final merged secondary data
obese_secondary_basedata = study.load_artifacts_data("/output_data/All_sex_Obese_secondary_basedata_Apr17")
value_cols = [c for c in obese_secondary_basedata.columns if 'Value' in c]

# Q1, Median, Q3, Min, Max
desc = obese_secondary_basedata[value_cols].describe(percentiles=[0.25, 0.5, 0.75])

result = desc.loc[['min', '25%', '50%', '75%', 'max']]
result.index = ['Min', 'Q1', 'Median', 'Q3', 'Max']

print(result.T.to_string())

In [ ]:
# Final merged primary data
obese_primary_basedata = study.load_artifacts_data("/output_data/All_sex_Obese_primary_basedata_Apr17")
value_cols = [c for c in obese_primary_basedata.columns if 'Value' in c]

# Q1, Median, Q3, Min, Max
desc = obese_primary_basedata[value_cols].describe(percentiles=[0.25, 0.5, 0.75])

result = desc.loc[['min', '25%', '50%', '75%', 'max']]
result.index = ['Min', 'Q1', 'Median', 'Q3', 'Max']

print(result.T.to_string())

In [ ]:
import pyspark.pandas as ps

t2dm_secondary_basedata = study.load_artifacts_data("/output_data/T2DM_female_secondary_basedata_Mar08")
value_cols = [c for c in t2dm_secondary_basedata.columns if 'Value' in c]

# Q1, Median, Q3, Min, Max
desc = t2dm_secondary_basedata[value_cols].describe(percentiles=[0.25, 0.5, 0.75])

result = desc.loc[['min', '25%', '50%', '75%', 'max']]
result.index = ['Min', 'Q1', 'Median', 'Q3', 'Max']

print(result.T.to_string())

In [ ]:
t2dm_primary_basedata = study.load_artifacts_data("/output_data/T2DM_female_primary_basedata_Mar08")
value_cols = [c for c in t2dm_primary_basedata.columns if 'Value' in c]

# Q1, Median, Q3, Min, Max
desc = t2dm_primary_basedata[value_cols].describe(percentiles=[0.25, 0.5, 0.75])

result = desc.loc[['min', '25%', '50%', '75%', 'max']]
result.index = ['Min', 'Q1', 'Median', 'Q3', 'Max']

print(result.T.to_string())

In [ ]:
import pyspark.pandas as ps

t2dm_secondary_basedata = study.load_artifacts_data("/output_data/T2DM_male_secondary_basedata_Mar09")
value_cols = [c for c in t2dm_secondary_basedata.columns if 'Value' in c]

# Q1, Median, Q3, Min, Max
desc = t2dm_secondary_basedata[value_cols].describe(percentiles=[0.25, 0.5, 0.75])

result = desc.loc[['min', '25%', '50%', '75%', 'max']]
result.index = ['Min', 'Q1', 'Median', 'Q3', 'Max']

print(result.T.to_string())

In [ ]:
t2dm_primary_basedata = study.load_artifacts_data("/output_data/T2DM_male_primary_basedata_Mar09")
value_cols = [c for c in t2dm_primary_basedata.columns if 'Value' in c]

# Q1, Median, Q3, Min, Max
desc = t2dm_primary_basedata[value_cols].describe(percentiles=[0.25, 0.5, 0.75])

result = desc.loc[['min', '25%', '50%', '75%', 'max']]
result.index = ['Min', 'Q1', 'Median', 'Q3', 'Max']

print(result.T.to_string())

## New column generation: Num of distinct antidiabetic drugs

In [ ]:
from pyspark.sql import functions as F

med_cols = ['HasInsulin', 'HasMetformin', 'HasSulfonylureas', 'HasThiazolidinedione', 'HasAlphaGlucosidaseInhibitors']

obese_secondary_basedata = study.load_artifacts_data("/output_data/All_sex_Obese_secondary_basedata_Apr17")
sdf = obese_secondary_basedata.to_spark()

sdf = sdf.withColumn(
    'NumAntidiabeticMed',
    sum([F.col(c).cast('int') for c in med_cols])
)

obese_secondary_basedata = sdf.to_pandas_on_spark()
# print(t2dm_secondary_basedata['NumAntidiabeticMed'].value_counts().sort_index())

In [ ]:
from pyspark.sql import functions as F

# Bool to int
bool_cols = [
    'HasEncounter', 'HasCvd', 'ExclT2d', 'ExclPriorDiabeticDrugs', 'ExclT1d', 'ExclGestDiab', 'ExclMody', 'ExclMedullaryThyroidCarcinoma',
    'ExclPancreaticCancer', 'ExclChronicPancreatitis', 'ExclEndStageRenalDisease', 'HasHospitalization', 'HasCongestiveHeartFailure',
    'HasCardiacArrhythmias', 'HasValvularHeartDisease', 'HasPulmonaryCirculationDisorders', 'HasPeripheralVascularDisorders',
    'HasChronicHypertensionWithoutComplications', 'HasChronicHypertensionWithComplications', 'HasParalysis', 'HasOtherNeurologicalDisorders', 'HasChronicPulmonaryDisease',
    'HasDiabetesUncomplicated', 'HasDiabetesComplicated', 'HasHyperthyroidism', 'HasRenalFailure', 'HasLiverDisease', 'HasPepticUlcerDisease',
    'HasHivAids', 'HasLymphoma', 'HasMetastaticCancer', 'HasSolidTumorWithoutMetastasis', 'HasRheumatoidArthritis', 'HasCoagulopathy',
    'HasObesityByConditionOrBmi30', 'HasWeightLossAndMalnutrition', 'HasFluidAndElectrolyteDisorders', 'HasBloodLossAnemia', 'HasDeficiencyAnemia',
    'HasAlcoholAbuse', 'HasDrugAbuse', 'HasPsychoses', 'HasDepressionDiagnosis', 'HasSmoking', 'HasDementia', 'HasHyperlipidemia',
    'HasParaplegiaAndHemiplegia', 'HasHemorrhagicOrIschemicStroke', 'HasOsteoarthritis', 'HasObstructiveSleepApnea', 'HasNafld', 'HasPolycysticOvarianSyndrome',
    'HasSchizophrenia', 'HasTraumaticBrainInjury', 'HasChronicObstructivePulmonaryDiseaseOrAsthma', 'HasCerebrovascularDisease', 'HasCoronaryArteryDisease',
    'HasPeripheralArterialDisease', 'HasAbdominalAorticAneurysm', 'HasAntiHypertensiveDrugHistory',
    'HasStrokeHistory', 'HasMiHistory', 'HasStatin', 'HasOtherLLT', 'HasAntiplatelet', 'HasAnticoagulant', 'HasInsulin',
    'HasMetformin', 'HasAceorArb', 'HasFinerenone', 'HasOtherAntiHypertensive', 'HasSulfonylureas', 'HasThiazolidinedione', 'HasAlphaGlucosidaseInhibitors',
]

# pandas-on-Spark to PySpark
df = obese_secondary_basedata.to_spark()

# Bool to int
for col in bool_cols:
    df = df.withColumn(col, F.col(col).cast("int"))

# Age recalculation
df = df.drop("InputEventId", "Age", "IsExpired")
df = df.withColumn("FirstMedTime", F.to_timestamp(F.col("FirstMedTime")))
df = df.withColumn("BirthDateTime", F.to_timestamp(F.col("BirthDateTime")))
df = df.withColumn(
    "Age",
    (F.datediff(F.col("FirstMedTime"), F.col("BirthDateTime")) / 365.25).cast("int")
)

# Timestamp floor to date
time_cols = [c for c in df.columns if 'Time' in c]
for col in time_cols:
    df = df.withColumn(col, F.date_trunc("day", F.col(col)).cast("date"))

# Drop MeasureDateTime and Unit columns
cols_to_drop = [c for c in df.columns if 'Unit' in c]
df = df.drop(*cols_to_drop)

# Elixhauser Score
weights = {
    "HasCongestiveHeartFailure": 7,
    "HasCardiacArrhythmias": 5,
    "HasValvularHeartDisease": -1,
    "HasPulmonaryCirculationDisorders": 4,
    "HasPeripheralVascularDisorders": 2,
    "HasChronicHypertensionWithoutComplications": 0,
    "HasChronicHypertensionWithComplications": 0,
    "HasParalysis": 7,
    "HasOtherNeurologicalDisorders": 6,
    "HasChronicPulmonaryDisease": 3,
    "HasDiabetesUncomplicated": 0,
    "HasDiabetesComplicated": 0,
    "HasHyperthyroidism": 0,
    "HasRenalFailure": 5,
    "HasLiverDisease": 11,
    "HasPepticUlcerDisease": 0,
    "HasHivAids": 0,
    "HasLymphoma": 9,
    "HasMetastaticCancer": 12,
    "HasSolidTumorWithoutMetastasis": 4,
    "HasRheumatoidArthritis": 0,
    "HasCoagulopathy": 3,
    "HasObesityByConditionOrBmi30": -4,
    "HasWeightLossAndMalnutrition": 6,
    "HasFluidAndElectrolyteDisorders": 5,
    "HasBloodLossAnemia": -2,
    "HasDeficiencyAnemia": -2,
    "HasAlcoholAbuse": 0,
    "HasDrugAbuse": -7,
    "HasPsychoses": 0,
    "HasDepressionDiagnosis": -3,
}

score_col = F.lit(0)
for col, w in weights.items():
    score_col = score_col + F.col(col) * F.lit(w)

df = df.withColumn("ElixhauserComorbidityScore", score_col)

# AgeGroup
df = df.withColumn(
    "AgeGroup",
    F.when((F.col("Age") >= 18) & (F.col("Age") <= 44), "18-44")
     .when((F.col("Age") >= 45) & (F.col("Age") <= 64), "45-64")
     .when(F.col("Age") >= 65, "65+")
     .otherwise(None)
)

# Null → "Unknown"
cols_to_unknown = ['Sex', 'Ethnicity', 'Race', 'CensusRegion', 'CensusDivision', 'StateOrProvince']
for col in cols_to_unknown:
    df = df.withColumn(col, F.coalesce(F.col(col).cast("string"), F.lit("Unknown")))

obese_secondary_basedata = df

study.save_artifacts_data(df=obese_secondary_basedata, path="/output_data/All_sex_Obese_secondary_basedata_Apr17_v2", data_type="parquet", mode="overwrite")

In [ ]:
df = study.load_artifacts_data('/output_data/All_sex_Obese_secondary_basedata_Apr17')
df.shape

In [ ]:
df2 = study.load_artifacts_data('/output_data/All_sex_Obese_secondary_basedata_Apr17_v2')
df2.shape

## Combine SDOH info

In [ ]:
%run _tr_s-rcyyfnog23pepk3szbfqahtygq

In [ ]:
dd = snapshot.get_data_dictionary()

In [ ]:
tr_initialize_helper_views(dd)
tr_initialize_theme_truveta()

In [ ]:
ps.sql("SELECT * FROM TR_SDOH_Attributes")

In [ ]:
%%sql
SELECT *
FROM TR_SDOH_Attributes

In [ ]:
sdoh_df_categorical = snapshot.get_sdoh_attribute_by_person_id(
    'Category',
    concept_ids=[3058440, 3058449, 2506706, 2506707, 2506710, 2506718, 3058862,
    3058863, 3001498, 3001494, 3058861, 3058444, 3058860, 2506713, 3001496, 3058864, 3058867])
sdoh_df_numerical = snapshot.get_sdoh_attribute_by_person_id(
    'Numeric',
    concept_ids=[2506701, 2506702, 2506703, 2506704, 2506705, 3058868, 3058865, 3058869,
    2506712, 3001497, 3001493, 2703601, 2506715, 2506717, 2506719, 3001495, 3058866])

In [ ]:
t2d_all_sex_primary = study.load_artifacts_data("/output_data/All_sex_Obese_primary_basedata_Apr17_v2")
t2d_all_sex_secondary = study.load_artifacts_data("/output_data/All_sex_Obese_secondary_basedata_Apr17_v2")

In [ ]:
t2d_secondary_sex_filtered = t2d_all_sex_secondary[t2d_all_sex_secondary['Sex'] == 1065405]
print(t2d_secondary_sex_filtered.shape)
t2d_secondary_merged = t2d_secondary_sex_filtered.merge(sdoh_df_categorical, on='PersonId', how='left')
t2d_secondary_merged = t2d_secondary_merged.merge(sdoh_df_numerical, on='PersonId', how='left')
print(t2d_secondary_merged.shape)

In [ ]:
t2d_secondary_merged = t2d_secondary_merged.drop_duplicates()

save_path = "/output_data/Obese_secondary_basedata_female_with_sdoh_Apr17"
study.save_artifacts_data(df=t2d_secondary_merged, path=save_path, data_type="parquet", mode="overwrite")

In [ ]:
t2d_primary_sex_filtered = t2d_all_sex_primary[t2d_all_sex_primary['Sex'] == 1065405]
print(t2d_primary_sex_filtered.shape)
t2d_primary_merged = t2d_primary_sex_filtered.merge(sdoh_df_categorical, on='PersonId', how='left')
t2d_primary_merged = t2d_primary_merged.merge(sdoh_df_numerical, on='PersonId', how='left')
print(t2d_primary_merged.shape)

In [ ]:
t2d_primary_merged = t2d_primary_merged.drop_duplicates()

save_path = "/output_data/Obese_primary_basedata_female_with_sdoh_Apr17"
study.save_artifacts_data(df=t2d_primary_merged, path=save_path, data_type="parquet", mode="overwrite")

In [ ]:
df1 = study.load_artifacts_data("/output_data/Obese_primary_basedata_female_with_sdoh_Apr17")
df2 = study.load_artifacts_data("/output_data/Obese_primary_basedata_male_with_sdoh_Apr17")
df3 = study.load_artifacts_data("/output_data/Obese_secondary_basedata_female_with_sdoh_Apr17")
df4 = study.load_artifacts_data("/output_data/Obese_secondary_basedata_male_with_sdoh_Apr17")

dm1 = ps.concat([df1, df2]).drop_duplicates()
dm2 = ps.concat([df3, df4]).drop_duplicates()

print(dm1.shape)
print(dm2.shape)

In [ ]:
cols_to_remove = [
    "HouseholdMembersCount",
    'EducationAttendedCollege',
    'ProspectEstimatedIncomeRange',
    'EstimatedAnnualIncome_24',
    'HouseholdElderlyMemberCount65+',
    'HouseholdAnnualIncomeRange',
    'EstimatedAnnualIncome_12',
    'EstimatedAnnualIncome',
    'WeightValue',
    'HeightValue',
    'Weight_MeasureDateTime',
    'Height_MeasureDateTime',
    'LpA_MeasureDateTime',
    'LipoproteinAValue',
    'BetaHcg_MeasureDateTime',
    'BetaHcgValue',
    'PlateletCount_MeasureDateTime',
    'PlateletCountValue',
    'CRP_MeasureDateTime',
    'CReactiveProteinValue'
]

categorical_cols_to_fill = [
    'HighestEduIndividual',
    'CollegePastPresent',
    'CurrAddrDwellType',
    'ShortLenofResBcEviction',
    'DistanceToCloseTies',
    'CurrentAddressOwnOrRent',
    'HealthManagementNonMotivationRiskCategory',
    'HealthcareCostRiskCategory',
    'HighestEduRelatives',
    'HighestEduHousehold',
    'MedicationNonAdherenceRiskCategory'
]

def clean_dataframe(df):
    existing_cols_to_remove = [col for col in cols_to_remove if col in df.columns]
    df = df.drop(columns=existing_cols_to_remove)

    # Null to Unknown
    for col in categorical_cols_to_fill:
        if col in df.columns:
            df[col] = df[col].fillna('Unknown')

    return df

primary_t2dm = clean_dataframe(dm1)
secondary_t2dm = clean_dataframe(dm2)

save_path1 = "/output_data/All_sex_Obese_primary_basedata_with_sdoh_Apr17_v2"
study.save_artifacts_data(df=primary_t2dm, path=save_path1, data_type="parquet", mode="overwrite")

save_path2 = "/output_data/All_sex_Obese_secondary_basedata_with_sdoh_Apr17_v2"
study.save_artifacts_data(df=secondary_t2dm, path=save_path2, data_type="parquet", mode="overwrite")

In [ ]:
df1 = study.load_artifacts_data('/output_data/All_sex_Obese_primary_basedata_with_sdoh_Apr17_v2')
df2 = study.load_artifacts_data('/output_data/All_sex_Obese_secondary_basedata_with_sdoh_Apr17_v2')

print(df1.shape)
print(df2.shape)

In [ ]:
# T2D secondary
missing_pct = df2.isna().sum() / len(df2) * 100
missing_pct = missing_pct.round(2).sort_values(ascending=False)
print(missing_pct)

In [ ]:
# T2D primary
missing_pct = df1.isna().sum() / len(df1) * 100
missing_pct = missing_pct.round(2).sort_values(ascending=False)
print(missing_pct)

### Data check

In [ ]:
t2d_primary = study.load_artifacts_data('/output_data/All_sex_Obese_primary_basedata_with_sdoh_Apr17_v2')
t2d_secondary = study.load_artifacts_data('/output_data/All_sex_Obese_secondary_basedata_with_sdoh_Apr17_v2')

print(t2d_primary.shape)
print(t2d_secondary.shape)

In [ ]:
import pyspark.pandas as ps
import numpy as np

cols = [
    'AddressChangeCountLast6Months',
    'AddressChangeCountLast12Months',
    'HouseholdMotorizedPropertyRegistrationsCount',
    'AddressChangeCountLast60Months',
    'FirstDegreeRelativesCount',
    'AddressChangeCountLast3Months',
    'IndividualMotorizedPropertyRegistrationsCount',
    'AddressChangeCountLast1Month',
    'CurrAddrMedianIncome',
    'RaAMmbrCnt',
    'CurrAddrLenOfRes',
    'EvictionAge',
    'HealthcareCostPercentileRank',
    'MedicationAdherenceRate',
    'HealthManagementMotivationLevel'
]

df_subset = t2d_primary[cols]
stats = df_subset.describe(percentiles=[0.25, 0.5, 0.75])

result = stats.loc[['min', '25%', '50%', '75%', 'max']].T

result = result.rename(columns={
    'min': 'Min',
    '25%': 'Q1(25%)',
    '50%': 'Median(50%)',
    '75%': 'Q3(75%)',
    'max': 'Max'
})

print(result.to_string())

In [ ]:
target_cols = ['CurrAddrMedianIncome', 'CurrAddrLenOfRes', 'RaAMmbrCnt']

for col in target_cols:
    t2d_primary[col] = t2d_primary[col].astype(float)
    t2d_primary[col] = t2d_primary[col].where(t2d_primary[col] >= 0, other=None)

t2d_primary = t2d_primary.drop(columns=['EvictionAge'])

t2d_primary['CalendarYear'] = t2d_primary['FirstMedTime'].dt.year

save_path = "/output_data/All_sex_Obese_primary_basedata_with_sdoh_Apr17_v3"
study.save_artifacts_data(df=t2d_primary, path=save_path, data_type="parquet", mode="overwrite")

## Add columns: YearsSinceVascEvent, hs-crp (Only for secondary cohort)

In [ ]:
t2d_secondary = study.load_artifacts_data('/output_data/All_sex_Obese_secondary_basedata_with_sdoh_Apr17_v3')
t2d_primary = study.load_artifacts_data('/output_data/All_sex_Obese_primary_basedata_with_sdoh_Apr17_v3')

all_first_vascular_event = study.load_artifacts_data('/output_data/All_sex_FirstVascularEvent')

In [ ]:
from pyspark.sql import functions as F

# 1. Convert to Spark DataFrames
t2d_secondary_spark = t2d_secondary.to_spark()
all_first_vascular_event_spark = all_first_vascular_event.to_spark().withColumnRenamed('PersonId', 'PersonId_event')

# 2. Join
t2d_secondary_spark = t2d_secondary_spark.join(
    all_first_vascular_event_spark.select('PersonId_event', 'IndexDateTime', 'firstVascularEventTime'),
    (t2d_secondary_spark.PersonId == all_first_vascular_event_spark.PersonId_event) &
    (t2d_secondary_spark.FirstMedTime == all_first_vascular_event_spark.IndexDateTime),
    how='left'
)

# 3. Drop extra columns
t2d_secondary_spark = t2d_secondary_spark.drop('PersonId_event', 'IndexDateTime')

# 4. Convert to timestamp
t2d_secondary_spark = t2d_secondary_spark.withColumn(
    'FirstMedTime_ts', F.to_timestamp(F.col('FirstMedTime'))
).withColumn(
    'firstVascularEventTime_ts', F.to_timestamp(F.col('firstVascularEventTime'))
)

# 5. Calculate YearsSinceVascEvent
t2d_secondary_spark = t2d_secondary_spark.withColumn(
    'YearsSinceVascEvent',
    F.round(
        F.datediff(F.col('FirstMedTime_ts'), F.col('firstVascularEventTime_ts')) / 365.25, 2
    )
)

# 6. Drop temporary timestamp columns
t2d_secondary_spark = t2d_secondary_spark.drop('FirstMedTime_ts', 'firstVascularEventTime_ts', 'firstVascularEventTime')

# 7. Convert back to pyspark.pandas
t2d_secondary = t2d_secondary_spark.to_pandas_on_spark()

In [ ]:
# ---------------------------------
# 0. HsCRP median dictionary
# ---------------------------------
hc_crp_median = {
    'CVD': {
        '<=40': {'M': 1.7, 'F': 2.0},
        '>40-50': {'M': 1.8, 'F': 1.5},
        '>50-60': {'M': 2.475, 'F': 2.3},
        '>60-70': {'M': 2.06, 'F': 2.025},
        '>70': {'M': 2.7, 'F': 2.4}
    },
    'CHD': {
        '<=40': {'M': 1.29, 'F': 1.3},
        '>40-50': {'M': 1.54, 'F': 2.15},
        '>50-60': {'M': 1.6, 'F': 2.06},
        '>60-70': {'M': 1.9, 'F': 2.0},
        '>70': {'M': 2.3, 'F': 2.46}
    },
    'PAD': {
        '<=40': {'M': 1.4, 'F': 8.19},
        '>40-50': {'M': 3.175, 'F': 4.495},
        '>50-60': {'M': 2.71, 'F': 3.92},
        '>60-70': {'M': 3.1, 'F': 2.43},
        '>70': {'M': 3.35, 'F': 3.035}
    },
    'AAA': {
        '<=40': {'M': 1.105, 'F': 0.8},
        '>40-50': {'M': 2.6, 'F': 4.2},
        '>50-60': {'M': 3.5, 'F': 3.35},
        '>60-70': {'M': 3.59, 'F': 2.7},
        '>70': {'M': 5.085, 'F': 2.95}
    }
}

t2d_secondary_spark = t2d_secondary.to_spark()

t2d_secondary_spark = t2d_secondary_spark.withColumn(
    "AgeGroup2",
    F.when(F.col("Age") <= 40, "<=40")
     .when(F.col("Age") <= 50, ">40-50")
     .when(F.col("Age") <= 60, ">50-60")
     .when(F.col("Age") <= 70, ">60-70")
     .otherwise(">70")
)

t2d_secondary_spark = t2d_secondary_spark.withColumn(
    "SexStr",
    F.when(F.col("Sex") == 1065405, "F")
     .when(F.col("Sex") == 1065406, "M")
)

t2d_secondary_spark = t2d_secondary_spark.withColumn(
    "DiseaseType",
    F.when(F.col("HasCerebrovascularDisease") == 1, "CVD")
     .when(F.col("HasCoronaryArteryDisease") == 1, "CHD")
     .when(F.col("HasPeripheralArterialDisease") == 1, "PAD")
     .when(F.col("HasAbdominalAorticAneurysm") == 1, "AAA")
     .otherwise("CVD")
)

hc_data = []
for disease, age_map in hc_crp_median.items():
    for age_group, sex_map in age_map.items():
        for sex, value in sex_map.items():
            hc_data.append((disease, age_group, sex, value))

hc_df = pd.DataFrame(hc_data, columns=["DiseaseType", "AgeGroup2", "SexStr", "MedianHsCRP"])
hc_df_spark = spark.createDataFrame(hc_df)

t2d_secondary_spark = t2d_secondary_spark.join(
    hc_df_spark,
    on=["DiseaseType", "AgeGroup2", "SexStr"],
    how="left"
)

t2d_secondary_spark = t2d_secondary_spark.withColumn(
    "HsCRPValue",
    F.when(F.col("HsCRPValue").isNull(), F.col("MedianHsCRP"))
     .otherwise(F.col("HsCRPValue"))
)

t2d_secondary_spark = t2d_secondary_spark.drop("MedianHsCRP", "AgeGroup2", "SexStr", "DiseaseType")
t2d_secondary = t2d_secondary_spark.to_pandas_on_spark()

save_path2 = "/output_data/All_sex_Obese_secondary_basedata_with_sdoh_Apr17_v4"
study.save_artifacts_data(df=t2d_secondary, path=save_path2, data_type="parquet", mode="overwrite")

## ICD-10 update on DSCI scores

In [ ]:
output_path_spark = study.get_output_path()

t2d_secondary = output_path_spark + "/output_data/All_sex_T2d_secondary_basedata_with_sdoh_Mar09_v4"
base_df = spark.read.parquet(t2d_secondary)
base_df = base_df.filter(base_df.Sex == 1065406)
base_df = base_df.dropDuplicates()
base_df.createOrReplaceTempView('tbl_basedata')

In [ ]:
output_path_spark = study.get_output_path()

t2d_primary = output_path_spark + "/output_data/All_sex_T2d_primary_basedata_with_sdoh_Mar09_v3"
base_df = spark.read.parquet(t2d_primary)
base_df = base_df.filter(base_df.Sex == 1065406)
base_df = base_df.dropDuplicates()
base_df.createOrReplaceTempView('tbl_basedata')

In [ ]:
# metabolic
ICD9_metabolic_score2 = ['249.1', '249.2', '249.3']
ICD10_metabolic_score1 = ['E08.00', 'E09.00', 'E10.00', 'E11.00', 'E13.00', 'E08.10', 'E09.10', 'E10.10', 'E11.10', 'E13.10', 'E08.649', 'E09.649', 'E10.649', 'E11.649', 'E13.649']
ICD10_metabolic_score2 = ['E08.01', 'E09.01', 'E10.01', 'E11.01', 'E13.01', 'E08.11', 'E09.11', 'E10.11', 'E11.11', 'E13.11', 'E08.641', 'E09.641', 'E10.641', 'E11.641', 'E13.641']

ICD9_metabolic_score2_codeset = snapshot.codeset("ICD9CM", 'selfAndDescendants', *ICD9_metabolic_score2)
ICD10_metabolic_score1_codeset = snapshot.codeset("ICD10CM", 'self', *ICD10_metabolic_score1)
ICD10_metabolic_score2_codeset = snapshot.codeset("ICD10CM", 'self', *ICD10_metabolic_score2)

metabolic_score2_codeset = ps.concat([ICD9_metabolic_score2_codeset, ICD10_metabolic_score2_codeset])

metabolic_score1_condition = snapshot.load_filtered_table('Condition', ICD10_metabolic_score1_codeset, apply_annulled_filter=True, view_name="tbl_metabolic_score1")
metabolic_score2_condition = snapshot.load_filtered_table('Condition', metabolic_score2_codeset, apply_annulled_filter=True, view_name="tbl_metabolic_score2")

In [ ]:
# neuropathy
ICD9_neuropathy_score1 = ['249.6']
ICD10_neuropathy_score1_all = ['E08.4', 'E09.4', 'E10.4', 'E11.4', 'E13.4', 'G56', 'G57', 'H49', 'M14.6', 'S04']
ICD10_neuropathy_score1_self = ['G90.09', 'G90.8', 'G90.9', 'G99.0', 'G60.9', 'G73.3', 'G90.01', 'I95.1', 'K31.84', 'K59.1', 'K31.9']

ICD9_neuropathy_score1_codeset = snapshot.codeset("ICD9CM", 'selfAndDescendants', *ICD9_neuropathy_score1)
ICD10_neuropathy_score1_all_codeset = snapshot.codeset("ICD10CM", 'selfAndDescendants', *ICD10_neuropathy_score1_all)
ICD10_neuropathy_score1_self_codeset = snapshot.codeset("ICD10CM", 'self', *ICD10_neuropathy_score1_self)

neuropathy_score1_codeset = ps.concat([ICD9_neuropathy_score1_codeset, ICD10_neuropathy_score1_all_codeset, ICD10_neuropathy_score1_self_codeset])
neuropathy_score1_condition = snapshot.load_filtered_table('Condition', neuropathy_score1_codeset, apply_annulled_filter=True, view_name="tbl_neuropathy_score1")

In [ ]:
# cerebrovascular
ICD10_cerebrovascular_score1 = ['G45']
ICD10_cerebrovascular_score2_all = ['I61', 'I63', 'I65', 'I66']
ICD10_cerebrovascular_score2_self = ['I67.81']

ICD10_cerebrovascular_score1_codeset = snapshot.codeset("ICD10CM", 'selfAndDescendants', *ICD10_cerebrovascular_score1)
ICD10_cerebrovascular_score2_all_codeset = snapshot.codeset("ICD10CM", 'selfAndDescendants', *ICD10_cerebrovascular_score2_all)
ICD10_cerebrovascular_score2_self_codeset = snapshot.codeset("ICD10CM", 'self', *ICD10_cerebrovascular_score2_self)

cerebrovascular_score1_condition = snapshot.load_filtered_table('Condition', ICD10_cerebrovascular_score1_codeset, apply_annulled_filter=True, view_name="tbl_cerebrovascular_score1")
cerebrovascular_score2_codeset = ps.concat([ICD10_cerebrovascular_score2_all_codeset, ICD10_cerebrovascular_score2_self_codeset])
cerebrovascular_score2_condition = snapshot.load_filtered_table('Condition', cerebrovascular_score2_codeset, apply_annulled_filter=True, view_name="tbl_cerebrovascular_score2")

In [ ]:
# retinopathy
ICD9_retinopathy_score1_all = ['249.5']
ICD9_retinopathy_score1_self = ['362.00', '362.01', '362.03', '362.04', '362.05', '362.06', '362.07']
ICD10_retinopathy_score1_all = ['E08.3', 'E09.3', 'E10.3', 'E11.3', 'E13.3', 'E08.34', 'E09.34', 'E10.34', 'E11.34', 'E13.34', 'E08.35', 'E09.35', 'E10.35', 'E11.35', 'E13.35',
                                'H35.0', 'H35.35', 'H35.6', 'H35.8']
ICD10_retinopathy_score1_self = ['H35.9']
ICD10_retinopathy_score2_all = ['H33', 'H54', 'H43.1', 'E08.34', 'E08.35', 'E09.34', 'E09.35', 'E10.34', 'E10.35', 'E11.34', 'E11.35', 'E13.34', 'E13.35']

ICD9_retinopathy_score1_all_codeset = snapshot.codeset("ICD9CM", 'selfAndDescendants', *ICD9_retinopathy_score1_all)
ICD9_retinopathy_score1_self_codeset = snapshot.codeset("ICD9CM", 'self', *ICD9_retinopathy_score1_self)
ICD10_retinopathy_score1_all_codeset = snapshot.codeset("ICD10CM", 'selfAndDescendants', *ICD10_retinopathy_score1_all)
ICD10_retinopathy_score1_self_codeset = snapshot.codeset("ICD10CM", 'self', *ICD10_retinopathy_score1_self)
ICD10_retinopathy_score2_all_codeset = snapshot.codeset("ICD10CM", 'selfAndDescendants', *ICD10_retinopathy_score2_all)

retinopathy_score1_codeset = ps.concat([ICD9_retinopathy_score1_all_codeset, ICD9_retinopathy_score1_self_codeset, ICD10_retinopathy_score1_all_codeset, ICD10_retinopathy_score1_self_codeset])

retinopathy_score1_condition = snapshot.load_filtered_table('Condition', retinopathy_score1_codeset, apply_annulled_filter=True, view_name="tbl_retinopathy_score1")
retinopathy_score2_condition = snapshot.load_filtered_table('Condition', ICD10_retinopathy_score2_all_codeset, apply_annulled_filter=True, view_name="tbl_retinopathy_score2")

In [ ]:
# Cardiovascular
ICD9_cardiovascular_score2_all = ['427.3', '427.4']
ICD9_cardiovascular_score2_self = ['427.5', '427.1']
ICD10_cardiovascular_score1_all = ['I24', 'I20', 'I70.0', 'I70.1', 'I70.3', 'I70.4', 'I70.5', 'I70.6', 'I70.7', 'I70.8', 'I70.9']
ICD10_cardiovascular_score1_self = ['I25.0', 'I25.1', 'I25.3', 'I25.4', 'I25.5', 'I25.6', 'I25.7', 'I25.8', 'I25.9', 'I70.20', 'I70.21', 'I70.22', 'I70.23', 'I70.24', 'I70.27', 'I70.28', 'I70.29']
ICD10_cardiovascular_score2_all = ['I21', 'I22', 'I23', 'I48', 'I46', 'I47', 'I49',' I50', 'I70.26', 'I71']
ICD10_cardiovascular_score2_self = ['I25.2', 'I70.25']

ICD9_cardiovascular_score2_all_codeset = snapshot.codeset("ICD9CM", 'selfAndDescendants', *ICD9_cardiovascular_score2_all)
ICD9_cardiovascular_score2_self_codeset = snapshot.codeset("ICD9CM", 'self', *ICD9_cardiovascular_score2_self)
ICD10_cardiovascular_score1_all_codeset = snapshot.codeset("ICD10CM", 'selfAndDescendants', *ICD10_cardiovascular_score1_all)
ICD10_cardiovascular_score1_self_codeset = snapshot.codeset("ICD10CM", 'self', *ICD10_cardiovascular_score1_self)
ICD10_cardiovascular_score2_all_codeset = snapshot.codeset("ICD10CM", 'selfAndDescendants', *ICD10_cardiovascular_score2_all)
ICD10_cardiovascular_score2_self_codeset = snapshot.codeset("ICD10CM", 'self', *ICD10_cardiovascular_score2_self)

cardiovascular_score1_codeset = ps.concat([ICD10_cardiovascular_score1_all_codeset, ICD10_cardiovascular_score1_self_codeset])
cardiovascular_score2_codeset = ps.concat([ICD9_cardiovascular_score2_all_codeset, ICD9_cardiovascular_score2_self_codeset, ICD10_cardiovascular_score2_all_codeset, ICD10_cardiovascular_score2_self_codeset])

cardiovascular_score1_condition = snapshot.load_filtered_table('Condition', cardiovascular_score1_codeset, apply_annulled_filter=True, view_name="tbl_cardiovascular_score1")
cardiovascular_score2_condition = snapshot.load_filtered_table('Condition', cardiovascular_score2_codeset, apply_annulled_filter=True, view_name="tbl_cardiovascular_score2")

In [ ]:
# Peripheral
ICD9_peripheral_score1_all = ['249.7']
ICD9_peripheral_score1_self = ['440.21', '443.81', '443.9']
ICD10_peripheral_score1_all = ['I70.21', 'S91.3']
ICD10_peripheral_score1_self = ['E08.51', 'E09.51', 'E10.51', 'E11.51', 'E13.51', 'E08.59', 'E09.59', 'E10.59', 'E11.59', 'E13.59', 'E08.621', 'E09.621', 'E10.621', 'E11.621', 'E13.621', 'I72.4', 'I73.89', 'I73.9']
ICD10_peripheral_score2_all = ['L97', 'E08.52', 'E09.52', 'E10.52', 'E11.52', 'E13.52']
ICD10_peripheral_score2_self = ['A48.0', 'I74.3', 'I96']

ICD9_peripheral_score1_all_codeset = snapshot.codeset("ICD9CM", 'selfAndDescendants', *ICD9_peripheral_score1_all)
ICD9_peripheral_score1_self_codeset = snapshot.codeset("ICD9CM", 'self', *ICD9_peripheral_score1_self)
ICD10_peripheral_score1_all_codeset = snapshot.codeset("ICD10CM", 'selfAndDescendants', *ICD10_peripheral_score1_all)
ICD10_peripheral_score1_self_codeset = snapshot.codeset("ICD10CM", 'self', *ICD10_peripheral_score1_self)
ICD10_peripheral_score2_all_codeset = snapshot.codeset("ICD10CM", 'selfAndDescendants', *ICD10_peripheral_score2_all)
ICD10_peripheral_score2_self_codeset = snapshot.codeset("ICD10CM", 'self', *ICD10_peripheral_score2_self)

peripheral_score1_codeset = ps.concat([ICD9_peripheral_score1_all_codeset, ICD9_peripheral_score1_self_codeset, ICD10_peripheral_score1_all_codeset, ICD10_peripheral_score1_self_codeset])
peripheral_score2_codeset = ps.concat([ICD10_peripheral_score2_all_codeset, ICD10_peripheral_score2_self_codeset])

peripheral_score1_condition = snapshot.load_filtered_table('Condition', peripheral_score1_codeset, apply_annulled_filter=True, view_name="tbl_peripheral_score1")
peripheral_score2_condition = snapshot.load_filtered_table('Condition', peripheral_score2_codeset, apply_annulled_filter=True, view_name="tbl_peripheral_score2")

In [ ]:
# nephropathy
ICD9_nephropathy_score1_all = ['249.4', '581']
ICD9_nephropathy_score1_self = ['585.1', '585.2', '585.3', '585.9']
ICD9_nephropathy_score2_self = ['585.4', '585.5', '585.6']
ICD10_nephropathy_score1_all = ['N00', 'N04', 'N03', 'N05']
ICD10_nephropathy_score1_self = ['N18.1', 'N18.2', 'N18.3', 'N18.9', 'E08.21', 'E09.21', 'E10.21', 'E11.21', 'E13.21', 'E08.22', 'E09.22', 'E10.22', 'E11.22', 'E13.22', 'E08.29', 'E09.29', 'E10.29', 'E11.29', 'E13.29']
ICD10_nephropathy_score2_self = ['N18.4', 'N18.5', 'N18.6', 'N19']

ICD9_nephropathy_score1_all_codeset = snapshot.codeset("ICD9CM", 'selfAndDescendants', *ICD9_nephropathy_score1_all)
ICD9_nephropathy_score1_self_codeset = snapshot.codeset("ICD9CM", 'self', *ICD9_nephropathy_score1_self)
ICD9_nephropathy_score2_self_codeset = snapshot.codeset("ICD9CM", 'self', *ICD9_nephropathy_score2_self)
ICD10_nephropathy_score1_all_codeset = snapshot.codeset("ICD10CM", 'selfAndDescendants', *ICD10_nephropathy_score1_all)
ICD10_nephropathy_score1_self_codeset = snapshot.codeset("ICD10CM", 'self', *ICD10_nephropathy_score1_self)
ICD10_nephropathy_score2_self_codeset = snapshot.codeset("ICD10CM", 'self', *ICD10_nephropathy_score2_self)

nephropathy_score1_codeset = ps.concat([ICD9_nephropathy_score1_all_codeset, ICD9_nephropathy_score1_self_codeset, ICD10_nephropathy_score1_all_codeset, ICD10_nephropathy_score1_self_codeset])
nephropathy_score2_codeset = ps.concat([ICD9_nephropathy_score2_self_codeset, ICD10_nephropathy_score2_self_codeset])

nephropathy_score1_condition = snapshot.load_filtered_table('Condition', nephropathy_score1_codeset, apply_annulled_filter=True, view_name="tbl_nephropathy_score1")
nephropathy_score2_condition = snapshot.load_filtered_table('Condition', nephropathy_score2_codeset, apply_annulled_filter=True, view_name="tbl_nephropathy_score2")

In [ ]:
dcsi_recalculate_result_query = """
    WITH base_range AS (
        SELECT PersonId,
               DATE_ADD(FirstMedTime, -730) AS StartDate,
               FirstMedTime                 AS EndDate
        FROM tbl_basedata
    ),
    all_scores AS (
        SELECT PersonId, 'Metabolic1'       AS ScoreType FROM tbl_metabolic_score1       JOIN base_range USING (PersonId) WHERE RecordedDateTime BETWEEN StartDate AND EndDate
        UNION ALL
        SELECT PersonId, 'Metabolic2'       AS ScoreType FROM tbl_metabolic_score2       JOIN base_range USING (PersonId) WHERE RecordedDateTime BETWEEN StartDate AND EndDate
        UNION ALL
        SELECT PersonId, 'Neuropathy1'      AS ScoreType FROM tbl_neuropathy_score1      JOIN base_range USING (PersonId) WHERE RecordedDateTime BETWEEN StartDate AND EndDate
        UNION ALL
        SELECT PersonId, 'Cerebrovascular1' AS ScoreType FROM tbl_cerebrovascular_score1 JOIN base_range USING (PersonId) WHERE RecordedDateTime BETWEEN StartDate AND EndDate
        UNION ALL
        SELECT PersonId, 'Cerebrovascular2' AS ScoreType FROM tbl_cerebrovascular_score2 JOIN base_range USING (PersonId) WHERE RecordedDateTime BETWEEN StartDate AND EndDate
        UNION ALL
        SELECT PersonId, 'Retinopathy1'     AS ScoreType FROM tbl_retinopathy_score1     JOIN base_range USING (PersonId) WHERE RecordedDateTime BETWEEN StartDate AND EndDate
        UNION ALL
        SELECT PersonId, 'Retinopathy2'     AS ScoreType FROM tbl_retinopathy_score2     JOIN base_range USING (PersonId) WHERE RecordedDateTime BETWEEN StartDate AND EndDate
        UNION ALL
        SELECT PersonId, 'Cardiovascular1'  AS ScoreType FROM tbl_cardiovascular_score1  JOIN base_range USING (PersonId) WHERE RecordedDateTime BETWEEN StartDate AND EndDate
        UNION ALL
        SELECT PersonId, 'Cardiovascular2'  AS ScoreType FROM tbl_cardiovascular_score2  JOIN base_range USING (PersonId) WHERE RecordedDateTime BETWEEN StartDate AND EndDate
        UNION ALL
        SELECT PersonId, 'Peripheral1'      AS ScoreType FROM tbl_peripheral_score1      JOIN base_range USING (PersonId) WHERE RecordedDateTime BETWEEN StartDate AND EndDate
        UNION ALL
        SELECT PersonId, 'Peripheral2'      AS ScoreType FROM tbl_peripheral_score2      JOIN base_range USING (PersonId) WHERE RecordedDateTime BETWEEN StartDate AND EndDate
        UNION ALL
        SELECT PersonId, 'Nephropathy1'     AS ScoreType FROM tbl_nephropathy_score1     JOIN base_range USING (PersonId) WHERE RecordedDateTime BETWEEN StartDate AND EndDate
        UNION ALL
        SELECT PersonId, 'Nephropathy2'     AS ScoreType FROM tbl_nephropathy_score2     JOIN base_range USING (PersonId) WHERE RecordedDateTime BETWEEN StartDate AND EndDate
    ),
    pivoted AS (
        SELECT PersonId,
               MAX(ScoreType = 'Metabolic1')       AS DsciMetabolic1,
               MAX(ScoreType = 'Metabolic2')       AS DsciMetabolic2,
               MAX(ScoreType = 'Neuropathy1')      AS DsciNeuropathy1,
               MAX(ScoreType = 'Cerebrovascular1') AS DsciCerebrovascular1,
               MAX(ScoreType = 'Cerebrovascular2') AS DsciCerebrovascular2,
               MAX(ScoreType = 'Retinopathy1')     AS DsciRetinopathy1,
               MAX(ScoreType = 'Retinopathy2')     AS DsciRetinopathy2,
               MAX(ScoreType = 'Cardiovascular1')  AS DsciCardiovascular1,
               MAX(ScoreType = 'Cardiovascular2')  AS DsciCardiovascular2,
               MAX(ScoreType = 'Peripheral1')      AS DsciPeripheral1,
               MAX(ScoreType = 'Peripheral2')      AS DsciPeripheral2,
               MAX(ScoreType = 'Nephropathy1')     AS DsciNephropathy1,
               MAX(ScoreType = 'Nephropathy2')     AS DsciNephropathy2
        FROM all_scores
        GROUP BY PersonId
    )
    SELECT
        b.PersonId,
        b.FirstMedTime,
        b.Sex,
        COALESCE(p.DsciMetabolic1,       false) AS DsciMetabolic1,
        COALESCE(p.DsciMetabolic2,       false) AS DsciMetabolic2,
        COALESCE(p.DsciNeuropathy1,      false) AS DsciNeuropathy1,
        COALESCE(p.DsciCerebrovascular1, false) AS DsciCerebrovascular1,
        COALESCE(p.DsciCerebrovascular2, false) AS DsciCerebrovascular2,
        COALESCE(p.DsciRetinopathy1,     false) AS DsciRetinopathy1,
        COALESCE(p.DsciRetinopathy2,     false) AS DsciRetinopathy2,
        COALESCE(p.DsciCardiovascular1,  false) AS DsciCardiovascular1,
        COALESCE(p.DsciCardiovascular2,  false) AS DsciCardiovascular2,
        COALESCE(p.DsciPeripheral1,      false) AS DsciPeripheral1,
        COALESCE(p.DsciPeripheral2,      false) AS DsciPeripheral2,
        COALESCE(p.DsciNephropathy1,     false) AS DsciNephropathy1,
        COALESCE(p.DsciNephropathy2,     false) AS DsciNephropathy2
    FROM tbl_basedata b
    LEFT JOIN pivoted p ON b.PersonId = p.PersonId
"""

dcsi_result = spark.sql(dcsi_recalculate_result_query)
dcsi_result.createOrReplaceTempView('tbl_dcsi_result')

In [ ]:
dcsi_result = dcsi_result.drop_duplicates()
save_path = "/output_data/T2d_Primary_dcsi_recalculation_male_Mar10_v2"
study.save_artifacts_data(df=dcsi_result, path=save_path, data_type="parquet", mode="overwrite")

## Final study cohort data generation

In [ ]:
t2d_sec_female_dcsi = study.load_artifacts_data('/output_data/T2d_Secondary_dcsi_recalculation_female_Mar10_v2')
t2d_sec_male_dcsi = study.load_artifacts_data('/output_data/T2d_Secondary_dcsi_recalculation_male_Mar10_v2')
t2d_pri_female_dcsi = study.load_artifacts_data('/output_data/T2d_Primary_dcsi_recalculation_female_Mar10_v2')
t2d_pri_male_dcsi = study.load_artifacts_data('/output_data/T2d_Primary_dcsi_recalculation_male_Mar10_v2')

t2d_sec_dcsi = ps.concat([t2d_sec_female_dcsi, t2d_sec_male_dcsi])
t2d_pri_dcsi = ps.concat([t2d_pri_female_dcsi, t2d_pri_male_dcsi])

t2d_sec_base = study.load_artifacts_data('/output_data/All_sex_T2d_secondary_basedata_with_sdoh_Mar09_v4')
t2d_pri_base = study.load_artifacts_data('/output_data/All_sex_T2d_primary_basedata_with_sdoh_Mar09_v3')

In [ ]:
dcsi_cols = [
    'DsciMetabolic1', 'DsciMetabolic2', 'DsciNeuropathy1',
    'DsciCerebrovascular1', 'DsciCerebrovascular2',
    'DsciRetinopathy1', 'DsciRetinopathy2',
    'DsciCardiovascular1', 'DsciCardiovascular2',
    'DsciPeripheral1', 'DsciPeripheral2',
    'DsciNephropathy1', 'DsciNephropathy2'
]

key_cols = ['PersonId', 'FirstMedTime', 'Sex']

dcsi_renamed = t2d_pri_dcsi[key_cols + dcsi_cols].rename(
    columns={col: col + '_v2' for col in dcsi_cols}
)

result = t2d_pri_base.merge(dcsi_renamed, on=key_cols, how='left')

In [ ]:
import pyspark.pandas as ps
ps.set_option('compute.ops_on_diff_frames', True)

def calc_dcsi_score(row):
    score = 0

    # Retinopathy
    ret1 = any([row['DcsiRetinopathy1'], row['DsciRetinopathy1_v2']])
    ret2 = any([row['DcsiRetinopathy2'], row['DsciRetinopathy2_v2']])
    score += 2 if ret2 else (1 if ret1 else 0)

    # Nephropathy
    nep1 = any([row['DcsiNephropathy1'], row['DsciNephropathy1_v2']])
    nep2 = any([row['DcsiNephropathy2'], row['DsciNephropathy2_v2']])
    score += 2 if nep2 else (1 if nep1 else 0)

    # Neuropathy
    neu = any([row['DcsiNeuropathy0'], row['DsciNeuropathy1_v2']])
    score += 1 if neu else 0

    # Cerebrovascular
    cer1 = any([row['DcsiCerebrovascular1'], row['DsciCerebrovascular1_v2']])
    cer2 = any([row['DcsiCerebrovascular2'], row['DsciCerebrovascular2_v2']])
    score += 2 if cer2 else (1 if cer1 else 0)

    # Cardiovascular
    card1 = any([row['DcsiCardiovascular1'], row['DsciCardiovascular1_v2']])
    card2 = any([row['DcsiCardiovascular2'], row['DsciCardiovascular2_v2']])
    score += 2 if card2 else (1 if card1 else 0)

    # Peripheral Vascular
    per1 = any([row['DcsiPeripheralVascular1'], row['DsciPeripheral1_v2']])
    per2 = any([row['DcsiPeripheralVascular2'], row['DsciPeripheral2_v2']])
    score += 2 if per2 else (1 if per1 else 0)

    # Metabolic
    met1 = any([row['DcsiMetabolic0'], row['DsciMetabolic1_v2']])
    met2 = row['DsciMetabolic2_v2']
    score += 2 if met2 else (1 if met1 else 0)

    return score

result['DcsiScore'] = result.apply(calc_dcsi_score, axis=1)

In [ ]:
drop_cols = [
    'DcsiRetinopathy0', 'DcsiRetinopathy1', 'DcsiRetinopathy2',
    'DcsiNephropathy0', 'DcsiNephropathy1', 'DcsiNephropathy2',
    'DcsiNeuropathy0',
    'DcsiCerebrovascular0', 'DcsiCerebrovascular1', 'DcsiCerebrovascular2',
    'DcsiCardiovascular0', 'DcsiCardiovascular1', 'DcsiCardiovascular2',
    'DcsiPeripheralVascular0', 'DcsiPeripheralVascular1', 'DcsiPeripheralVascular2',
    'DcsiMetabolic0', 'DsciMetabolic1_v2', 'DsciMetabolic2_v2', 'DsciNeuropathy1_v2',
    'DsciCerebrovascular1_v2', 'DsciCerebrovascular2_v2', 'DsciRetinopathy1_v2',
    'DsciRetinopathy2_v2', 'DsciCardiovascular1_v2', 'DsciCardiovascular2_v2',
    'DsciPeripheral1_v2', 'DsciPeripheral2_v2', 'DsciNephropathy1_v2',
    'DsciNephropathy2_v2', 'HsCRP_MeasureDateTime', 'HsCRPValue'
]

result = result.drop(columns=drop_cols)

save_path = "/output_data/T2D_Primary_basedata_Mar10"
study.save_artifacts_data(df=result, path=save_path, data_type="parquet", mode="overwrite")

## Last data sanity check

In [ ]:
df1 = study.load_artifacts_data('/output_data/Obese_Secondary_basedata_Apr17')
df2 = study.load_artifacts_data('/output_data/Obese_Primary_basedata_Apr17')
df3 = study.load_artifacts_data('/output_data/All_sex_Obese_secondary_basedata_Apr17')
df4 = study.load_artifacts_data('/output_data/All_sex_Obese_primary_basedata_Apr17')

print(df1.shape)
print(df2.shape)
print(df3.shape)
print(df4.shape)

In [ ]:
value_cols = [c for c in df1.columns if 'Value' in c]

# Q1, Median, Q3, Min, Max
desc = df1[value_cols].describe(percentiles=[0.25, 0.5, 0.75])

result = desc.loc[['min', '25%', '50%', '75%', 'max']]
result.index = ['Min', 'Q1', 'Median', 'Q3', 'Max']

print(result.T.to_string())

In [ ]:
import pyspark.pandas as ps
import numpy as np

cols = [
    'AddressChangeCountLast6Months',
    'AddressChangeCountLast12Months',
    'HouseholdMotorizedPropertyRegistrationsCount',
    'AddressChangeCountLast60Months',
    'FirstDegreeRelativesCount',
    'AddressChangeCountLast3Months',
    'IndividualMotorizedPropertyRegistrationsCount',
    'AddressChangeCountLast1Month',
    'CurrAddrMedianIncome',
    'RaAMmbrCnt',
    'CurrAddrLenOfRes',
    'HealthcareCostPercentileRank',
    'MedicationAdherenceRate',
    'HealthManagementMotivationLevel'
]

df_subset = df1[cols]
stats = df_subset.describe(percentiles=[0.25, 0.5, 0.75])

result = stats.loc[['min', '25%', '50%', '75%', 'max']].T

result = result.rename(columns={
    'min': 'Min',
    '25%': 'Q1(25%)',
    '50%': 'Median(50%)',
    '75%': 'Q3(75%)',
    'max': 'Max'
})

print(result.to_string())

In [ ]:
missing_pct = df2.isna().sum() / len(df2) * 100
missing_pct = missing_pct.round(2).sort_values(ascending=False)
print(missing_pct)

In [ ]:
value_cols = [c for c in df2.columns if 'Value' in c]

# Q1, Median, Q3, Min, Max
desc = df2[value_cols].describe(percentiles=[0.25, 0.5, 0.75])

result = desc.loc[['min', '25%', '50%', '75%', 'max']]
result.index = ['Min', 'Q1', 'Median', 'Q3', 'Max']

print(result.T.to_string())

In [ ]:
cols = [
    'AddressChangeCountLast6Months',
    'AddressChangeCountLast12Months',
    'HouseholdMotorizedPropertyRegistrationsCount',
    'AddressChangeCountLast60Months',
    'FirstDegreeRelativesCount',
    'AddressChangeCountLast3Months',
    'IndividualMotorizedPropertyRegistrationsCount',
    'AddressChangeCountLast1Month',
    'CurrAddrMedianIncome',
    'RaAMmbrCnt',
    'CurrAddrLenOfRes',
    'HealthcareCostPercentileRank',
    'MedicationAdherenceRate',
    'HealthManagementMotivationLevel'
]

df_subset = df2[cols]
stats = df_subset.describe(percentiles=[0.25, 0.5, 0.75])

result = stats.loc[['min', '25%', '50%', '75%', 'max']].T

result = result.rename(columns={
    'min': 'Min',
    '25%': 'Q1(25%)',
    '50%': 'Median(50%)',
    '75%': 'Q3(75%)',
    'max': 'Max'
})

print(result.to_string())

In [ ]:
missing_pct = df2.isna().sum() / len(df2) * 100
missing_pct = missing_pct.round(2).sort_values(ascending=False)
print(missing_pct)

## For consort diagram patient counts

In [ ]:
# from pyspark.sql.functions import col, months_between

# # T2D Cohort generation
# basedata = study.load_artifacts_data("/output_data/T2DM_male_basedata_Mar09")
# basedata_spark = basedata.to_spark()

# base_filtered_df = basedata_spark.filter(
#     (col("ExclT2d") == 'true') &
#     ((months_between(col("FirstMedTime"), col("BirthDateTime")) / 12) >= 18) &
#     (col("FirstMedGroup").isin('glp', 'SGLT', 'DPP4')) &
#     (col("FirstMedTime").isNotNull()) &
#     (col("HasEncounter") == 'true') &
#     # exclusion criteria
#     (col("ExclT1d") == 'false') &
#     (col("ExclGestDiab") == 'false') &
#     (col("ExclMody") == 'false') &
#     (col("ExclMedullaryThyroidCarcinoma") == 'false') &
#     (col("ExclPancreaticCancer") == 'false') &
#     (col("ExclChronicPancreatitis") == 'false') &
#     (col("ExclEndStageRenalDisease") == 'false')
# )

# # HasCvd == true
# cvd_df = base_filtered_df.filter(col("HasCvd") == 'true').drop_duplicates()

# # HasCvd == false
# # no_cvd_df = base_filtered_df.filter(col("HasCvd") == 'false').drop_duplicates()

In [ ]:
from pyspark.sql.functions import col, months_between, countDistinct

# T2D Cohort generation - Male + Female combined
basedata_male = study.load_artifacts_data("/output_data/T2DM_male_basedata_Mar09")
basedata_female = study.load_artifacts_data("/output_data/T2DM_Female_basedata_Mar08")

basedata_male_spark = basedata_male.to_spark()
basedata_female_spark = basedata_female.to_spark()

# Concat and drop duplicates
basedata_spark = basedata_male_spark.union(basedata_female_spark).dropDuplicates()

base_filtered_df = basedata_spark.filter(
    (col("FirstMedGroup").isin('glp', 'SGLT', 'DPP4')) &
    (col("FirstMedTime").isNotNull()) &
    (col("ExclT2d") == 'true') &
    ((months_between(col("FirstMedTime"), col("BirthDateTime")) / 12) >= 18) &
    (col("HasEncounter") == 'true') &
    # exclusion criteria
    (col("ExclT1d") == 'false') &
    (col("ExclGestDiab") == 'false') &
    (col("ExclMody") == 'false') &
    (col("ExclMedullaryThyroidCarcinoma") == 'false') &
    (col("ExclPancreaticCancer") == 'false') &
    (col("ExclChronicPancreatitis") == 'false') &
    (col("ExclEndStageRenalDisease") == 'false') &
    (col("HasCvd") == 'true')
)

base_filtered_df.select(countDistinct("PersonId")).collect()[0][0]

In [ ]:
base_filtered_df.groupBy("FirstMedGroup").agg(countDistinct("PersonId").alias("unique_persons")).show()

In [ ]:
from pyspark.sql.functions import col, months_between, countDistinct

# OB Cohort generation - Male + Female combined
basedata_male = study.load_artifacts_data("/output_data/T2DM_male_basedata_Mar09")
basedata_female = study.load_artifacts_data("/output_data/T2DM_Female_basedata_Mar08")

basedata_male_spark = basedata_male.to_spark()
basedata_female_spark = basedata_female.to_spark()

# Concat and drop duplicates
basedata_spark = basedata_male_spark.union(basedata_female_spark).dropDuplicates()

base_filtered_df = basedata_spark.filter(
    (col("FirstMedGroup").isin('glp', 'NB', 'PT')) &
    (col("FirstMedTime").isNotNull()) &
    (col("BmiValue") >= 27) &
    ((months_between(col("FirstMedTime"), col("BirthDateTime")) / 12) >= 18) &
    (col("ExclT2d") == 'false') &
    (col("ExclPriorDiabeticDrugs") == 'false') &
    (col("HasEncounter") == 'true') &
    # exclusion criteria
    (col("ExclT1d") == 'false') &
    (col("ExclGestDiab") == 'false') &
    (col("ExclMody") == 'false') &
    (col("ExclMedullaryThyroidCarcinoma") == 'false') &
    (col("ExclPancreaticCancer") == 'false') &
    (col("ExclChronicPancreatitis") == 'false') &
    (col("ExclEndStageRenalDisease") == 'false')
)

base_filtered_df.select(countDistinct("PersonId")).collect()[0][0]

In [ ]:
base_filtered_df = basedata_spark.filter(
    (col("FirstMedGroup").isin('glp', 'NB', 'PT')) &
    (col("FirstMedTime").isNotNull()) &
    (col("BmiValue") >= 27) &
    ((months_between(col("FirstMedTime"), col("BirthDateTime")) / 12) >= 18) &
    (col("HasEncounter") == 'true') &
    # exclusion criteria
    (col("ExclT1d") == 'false') &
    (col("ExclGestDiab") == 'false') &
    (col("ExclMody") == 'false') &
    (col("ExclMedullaryThyroidCarcinoma") == 'false') &
    (col("ExclPancreaticCancer") == 'false') &
    (col("ExclChronicPancreatitis") == 'false') &
    (col("ExclEndStageRenalDisease") == 'false') &
    (col("HasCvd") == 'true')
)